# Voz de Asuka para el directo (RVC)
Entrena el modelo que cambia **tu voz por la de Asuka en tiempo real** (Applio).

**Antes de pulsar «Run all»:** a la derecha, *Session options* → **Accelerator: GPU T4 x2** e **Internet: on**. En *Input*, añade el dataset **voz-asuka-datos** (el mismo de la voz de Athena; si no está, súbelo desde `voz-asuka-datos.zip` de esta carpeta).

Tarda unas 2 horas. Al final, en *Output*, descarga **asuka_voz_rvc.zip**.

In [ ]:
import glob, os
wavs = sorted(glob.glob('/kaggle/input/**/*.wav', recursive=True))
print(len(wavs), 'audios'); assert len(wavs) > 50, 'Falta el dataset: añade voz-asuka-datos en Input'
os.makedirs('/kaggle/working/dataset', exist_ok=True)
import shutil
for i, w in enumerate(wavs):
    shutil.copy(w, f'/kaggle/working/dataset/{i:04d}.wav')
print('copiados')

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/IAHispano/Applio.git
%cd Applio
# torch ya viene en Kaggle: se instala el resto
!grep -v -E '^(torch|torchaudio)==' requirements-engine.txt > req.txt
!pip install -q -r req.txt
!python core.py prerequisites --pretraineds-hifigan --models --no-exe

In [ ]:
!python core.py preprocess --model-name asuka --dataset-path /kaggle/working/dataset --sample-rate 40000 --cut-preprocess Automatic

In [ ]:
!python core.py extract --model-name asuka --f0-method rmvpe --gpu 0 --sample-rate 40000 --embedder-model contentvec --include-mutes 2 --cpu-cores 4

In [ ]:
!python core.py train --model-name asuka --sample-rate 40000 --total-epoch 300 --save-every-epoch 50 --save-only-latest --batch-size 8 --gpu 0 --pretrained

In [ ]:
!python core.py index --model-name asuka --index-algorithm Auto

In [ ]:
import glob, os, zipfile
d = '/kaggle/working/Applio/logs/asuka'
pth = sorted(glob.glob(f'{d}/asuka*.pth') + glob.glob('/kaggle/working/Applio/logs/asuka*.pth'), key=os.path.getmtime)
idx = sorted(glob.glob(f'{d}/*.index'), key=os.path.getmtime)
print(pth[-3:], idx[-1:])
with zipfile.ZipFile('/kaggle/working/asuka_voz_rvc.zip', 'w') as z:
    z.write(pth[-1], 'asuka.pth')
    z.write(idx[-1], 'asuka.index')
print('LISTO: descarga asuka_voz_rvc.zip en Output')

In [ ]:
# prueba: una frase convertida a su voz
!python core.py infer --input-path /kaggle/working/dataset/0001.wav --output-path /kaggle/working/prueba_asuka.wav --pth-path $(ls -t /kaggle/working/Applio/logs/asuka/asuka*.pth | head -1) --index-path $(ls -t /kaggle/working/Applio/logs/asuka/*.index | head -1) || echo 'la prueba ha fallado, pero el modelo del zip vale'